# Gaussian Beam Analysis of SMA Optics

In [1]:
import numpy as np
import scipy as sp
import matplotlib.pyplot as pp

import gaussianbeam as gb
mm, GHz, pi = gb.mm, gb.GHz, gb.pi

## Nominal System

Scott's analysis works outwards from the Virtual Feed behind L1

In [ ]:
# Create the input beam to the beam waveguide
virtualFeed = gb.GaussianBeam()
virtualFeed.setFrequency(230e9)

virtualFeed.R = -1291.8*mm
virtualFeed.w = 30.25*mm

virtualFeed.w0 = virtualFeed.calcW0(wavelength=virtualFeed.wavelength, w=virtualFeed.w, R=virtualFeed.R)
virtualFeed.z = virtualFeed.calcZ(wavelength=virtualFeed.wavelength, w0=virtualFeed.w0, R=virtualFeed.R)

# Virtual Feed is located 780 mm behind L1, so get z and add 780 mm to get beam properties at L1
L1 = virtualFeed.copy()
L1.z = L1.z + 780*mm

In [3]:
print("Virtual Feed waist properties: w0 = {:.3f} mm, zc = {:.3f} mm".format(virtualFeed.w0/mm, virtualFeed.zc/mm))
print("Virtual Feed waist distance from L1: z = {:.3f} m".format(L1.z))
print("Beam properties at L1: w = {:.3f} mm, R = {:.3f} m".format(L1.w/mm, L1.R))

Virtual Feed waist properties: w0 = 11.003 mm, zc = 437.684 mm
Virtual Feed waist distance from L1: z = -0.341 m
Beam properties at L1: w = 13.946 mm, R = -0.903 m


M6 is a simple flat fold mirror at 1761 mm from L1

In [4]:
M6 = L1.copy()
M6.z = M6.z + 1761*mm
print("Beam properties at M6: w = {:.3f} mm, R = {:.3f} m".format(M6.w/mm, M6.R))

Beam properties at M6: w = 37.357 mm, R = 1.555 m


M5 is an elliptical mirror with an effective focal length of 726.565 mm, positioned 400 mm from M6

In [5]:
M5 = gb.ThinLens(-726.565*mm)
inputBeamM5 = M6.copy()
inputBeamM5.z = M6.z + 400*mm
M5.inputBeam = inputBeamM5
print("Beam properties at M5: w = {:.3f} mm, R = {:.3f} m".format(inputBeamM5.w/mm, inputBeamM5.R))

Beam properties at M5: w = 47.060 mm, R = 1.925 m


In [6]:
outputBeamM5 = M5.outputBeam
print("M5 output beam properties at M5: w = {:.3f} mm, R = {:.3f} m".format(outputBeamM5.w/mm, outputBeamM5.R))
print("M5 output waist properties: w0 = {:.3f} mm, zc = {:.3f} mm".format(outputBeamM5.w0/mm, outputBeamM5.zc/mm))
print("M5 output waist distance from M5: z = {:.3f} m".format(outputBeamM5.z))

M5 output beam properties at M5: w = 47.060 mm, R = -1.167 m
M5 output waist properties: w0 = 6.787 mm, zc = 166.536 mm
M5 output waist distance from M5: z = -1.143 m


M4 is an elliptical mirror with an effective focal length of 309.784 mm, positioned 1295.64 mm from M5

In [7]:
M4 = gb.ThinLens(-309.784*mm)
inputBeamM4 = outputBeamM5.copy()
inputBeamM4.z = outputBeamM5.z + 1295.64*mm
M4.inputBeam = inputBeamM4

print("Beam properties at M4: w = {:.3f} mm, R = {:.3f} m".format(inputBeamM4.w/mm, inputBeamM4.R))

Beam properties at M4: w = 9.216 mm, R = 0.334 m


In [8]:
outputBeamM4 = M4.outputBeam
print("M4 output beam properties at M4: w = {:.3f} mm, R = {:.3f} m".format(outputBeamM4.w/mm, outputBeamM4.R))
print("M4 output waist properties: w0 = {:.3f} mm, zc = {:.3f} mm".format(outputBeamM4.w0/mm, outputBeamM4.zc/mm))
print("M4 output waist distance from M4: z = {:.3f} m".format(-outputBeamM4.z))

M4 output beam properties at M4: w = 9.216 mm, R = -4.229 m
M4 output waist properties: w0 = 9.192 mm, zc = 305.470 mm
M4 output waist distance from M4: z = 0.022 m


M3 is a flat mirror 435.1 mm in front of M4

In [9]:
outputBeamM4.z = outputBeamM4.z + 435.1*mm
print("Beam properties at M3 (z={:.3f} mm) : w = {:.3f} mm, R = {:.3f} m".format(outputBeamM4.z/mm, outputBeamM4.w/mm, outputBeamM4.R))

Beam properties at M3 (z=412.918 mm) : w = 15.456 mm, R = 0.639 m


M2 is a hyperbolic secondary mirror with its rim 4.466 m in front of M3 and 4901.1 mm in front of M4.

In [10]:
outputBeamM4.z = outputBeamM4.z + 4466*mm
print("Beam properties at M2 rim (z={:.3f} mm) : w = {:.3f} mm, R = {:.3f} m".format(outputBeamM4.z/mm, outputBeamM4.w/mm, outputBeamM4.R))

Beam properties at M2 rim (z=4878.918 mm) : w = 147.101 mm, R = 4.898 m


In [11]:
print("Edge taper at M2 rim: Te = {:.3f} dB".format(outputBeamM4.edgeTaperDB(175*mm)))

Edge taper at M2 rim: Te = -12.293 dB


## Continuation to Primary

Scott's analysis stops when it illuminates M2, calculating the edge taper at the M2 rim surface.  For comparison with alternative designs, we continue the analysis to the Primary mirror

M2 is a convex hyperbolic secondary mirror with its rim 4898.010 in front of the Cass focus, its vertex 4847.489 mm in front of the Cass focus and in front of M4, with an effective focal length of -141.201 mm.

In [12]:
M2 = gb.ThinLens(-141.201*mm)
inputBeamM2 = outputBeamM4.copy()
inputBeamM2.z = outputBeamM4.z - (4898.01-4847.489)*mm
M2.inputBeam = inputBeamM2

print("Beam properties at M2 vertex (z={:.3f} mm) : w = {:.3f} mm, R = {:.3f} m".format(inputBeamM2.z/mm, inputBeamM2.w/mm, inputBeamM2.R))

Beam properties at M2 vertex (z=4828.397 mm) : w = 145.583 mm, R = 4.848 m


In [13]:
outputBeamM2 = M2.outputBeam
print("M2 output waist properties: w0 = {:.3f} mm, zc = {:.3f} mm".format(outputBeamM2.w0/mm, outputBeamM2.zc/mm))
print("M2 output waist distance from M2: z = {:.3f} mm".format(-outputBeamM2.z/mm))

M2 output waist properties: w0 = 0.276 mm, zc = 0.276 mm
M2 output waist distance from M2: z = 145.437 mm


The primary is a concave parabolic mirror with a focal length of 2520 mm, and its focal point 145.437 mm behind the secondary vertex.

In [14]:
M1 = gb.ThinLens(2520*mm)
inputBeamM1 = outputBeamM2.copy()
inputBeamM1.z = outputBeamM2.z + 2520*mm - 145.437*mm
M1.inputBeam = inputBeamM1

print("Beam properties at M1: w = {:.3f} mm, R = {:.3f} m".format(inputBeamM1.w/mm, inputBeamM1.R))

Beam properties at M1: w = 2231.371 mm, R = 2.229 m


In [15]:
outputBeamM1 = M1.outputBeam
print("M1 output waist properties: w0 = {:.3f} mm, zc = {:.3f} mm".format(outputBeamM1.w0/mm, outputBeamM1.zc/mm))
print("M1 output waist distance from M1: z = {:.3f} m".format(-outputBeamM1.z))

M1 output waist properties: w0 = 0.147 mm, zc = 0.078 mm
M1 output waist distance from M1: z = -1.183 m
